In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from piano_tiles_ai.hand_data import load_split
from piano_tiles_ai.hand_model import train_hand_model

DATA_DIR = Path("../data/hands/piano-midi-de")
train_pieces = load_split(DATA_DIR, "train")
val_pieces = load_split(DATA_DIR, "val")
print(len(train_pieces), "training pieces,", len(val_pieces), "validation pieces")

model, history = train_hand_model(train_pieces, val_pieces, epochs=30)

In [ ]:
epochs = [h["epoch"] for h in history]

fig, ax_loss = plt.subplots(figsize=(10, 4))
ax_loss.plot(epochs, [h["train_loss"] for h in history], color="tab:blue")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Training loss", color="tab:blue")

ax_acc = ax_loss.twinx()
ax_acc.plot(epochs, [h["val_accuracy"] for h in history], color="tab:orange")
ax_acc.axhline(0.924, color="gray", linestyle="--")
ax_acc.set_ylabel("Validation accuracy", color="tab:orange")

ax_loss.set_title("Learning curves (dashed line: tuned clustering baseline)")
plt.show()

In [ ]:
Path("../models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "../models/hand_lstm.pt")
print(f"Best validation accuracy: {max(h['val_accuracy'] for h in history):.1%}")

LSTM vs TRANSFORMER

In [ ]:
from piano_tiles_ai.hand_model import HandLSTM, HandTransformer

EPOCHS = 50

lstm, lstm_history = train_hand_model(train_pieces, val_pieces, model=HandLSTM(), epochs=EPOCHS)
transformer, transformer_history = train_hand_model(train_pieces, val_pieces, model=HandTransformer(), epochs=EPOCHS)


def n_parameters(model):
    return sum(p.numel() for p in model.parameters())


for name, model_, history_ in [("LSTM", lstm, lstm_history), ("Transformer", transformer, transformer_history)]:
    best = max(h["val_accuracy"] for h in history_)
    print(f"{name:12s} {n_parameters(model_):7d} parameters   best val accuracy {best:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for name, history_ in [("LSTM", lstm_history), ("Transformer", transformer_history)]:
    ax.plot([h["epoch"] for h in history_], [h["val_accuracy"] for h in history_], label=name)
ax.axhline(0.924, color="gray", linestyle="--", label="clustering baseline")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0.85, 1.0)
ax.legend()
ax.set_title("LSTM vs Transformer on the validation split")
plt.show()

In [ ]:
torch.save(lstm.state_dict(), "../models/hand_lstm.pt")
torch.save(transformer.state_dict(), "../models/hand_transformer.pt")

## Final evaluation on the test split

In [ ]:
import torch

from piano_tiles_ai.hand_model import HandLSTM, HandTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

lstm = HandLSTM()
lstm.load_state_dict(torch.load("../models/hand_lstm.pt", map_location=device))
lstm.to(device)

transformer = HandTransformer()
transformer.load_state_dict(torch.load("../models/hand_transformer.pt", map_location=device))
transformer.to(device)

In [ ]:
from functools import partial

from piano_tiles_ai.evaluation import evaluate_method
from piano_tiles_ai.hand_data import load_split
from piano_tiles_ai.hand_model import assign_hands_with_model
from piano_tiles_ai.hands import assign_hands_by_clustering, assign_hands_by_split

test_pieces = load_split(DATA_DIR, "test")
print(len(test_pieces), "test pieces,", sum(len(p) for p in test_pieces), "notes\n")

methods = {
    "Split at middle C": assign_hands_by_split,
    "Clustering (1.0 s)": partial(assign_hands_by_clustering, window=1.0, min_spread=12),
    "Transformer": partial(assign_hands_with_model, model=transformer),
    "BiLSTM": partial(assign_hands_with_model, model=lstm),
}

print(f"{'Method':20s} {'Accuracy':>9s} {'Left P':>8s} {'Left R':>8s} {'Right P':>8s} {'Right R':>8s}")
for name, method in methods.items():
    s = evaluate_method(method, test_pieces)
    print(f"{name:20s} {s.accuracy:9.1%} {s.left_precision:8.1%} {s.left_recall:8.1%} "
          f"{s.right_precision:8.1%} {s.right_recall:8.1%}")